# Predicting Bitcoin Price Direction Using Machine Learning
### TEC411_MBA — Problem-Solving with AI | Wittenborg University of Applied Sciences

**Student:** Albert Okpala | **Student Number:** s1184857
**Date:** June 2026

This notebook applies four machine learning methods (Logistic Regression, Random Forest, XGBoost, and LSTM) to predict next-day Bitcoin price direction (up/down), using engineered technical indicators. The structure follows the TEC411 assignment brief: data preparation, model application (3–5 ML methods), evaluation, and visualization.

**How to run:** Click `Runtime > Run all` in Google Colab. Each section is self-contained and runs top to bottom.

## 1. Setup: Install and Import Libraries

In [ ]:
# Install packages not pre-installed in Colab (xgboost is usually present, but we pin versions for reproducibility)
!pip install yfinance xgboost scikit-learn tensorflow --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)

import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Reproducibility
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries loaded successfully.")

## 2. Data Collection

Two options are provided. **Option A** (recommended, no setup needed) pulls live historical Bitcoin data directly via the `yfinance` library — no Kaggle account or file upload required. **Option B** shows how to load a Kaggle CSV if you prefer to cite a specific Kaggle dataset in your report (e.g., "Bitcoin Historical Data" by mczielinski).

Use Option A unless your report specifically references a named Kaggle dataset.

In [ ]:
# ── OPTION A: Fetch data directly (recommended) ──────────────────────────
import yfinance as yf

df = yf.download("BTC-USD", start="2018-01-01", end="2026-06-01", progress=False)
df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
df.columns = ["open", "high", "low", "close", "volume"]
df.dropna(inplace=True)

print(f"Data shape: {df.shape}")
df.tail()

In [ ]:
# ── OPTION B: Load from Kaggle CSV instead (uncomment to use) ────────────
# 1. Download "Bitcoin Historical Data" from Kaggle: https://www.kaggle.com/datasets/mczielinski/bitcoin-historical-data
# 2. Upload the CSV to Colab (left sidebar > Files > Upload), then run:

# from google.colab import files
# uploaded = files.upload()  # select your CSV when prompted
# df = pd.read_csv("bitcoin_historical.csv", parse_dates=["Date"], index_col="Date")
# df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
# df.columns = ["open", "high", "low", "close", "volume"]
# df.dropna(inplace=True)

## 3. Feature Engineering

We construct technical indicators commonly used in the literature reviewed in the report (SMA, RSI, MACD, Bollinger Bands, rolling volatility). These become the model's input features (X). The target (y) is binary: 1 if tomorrow's close is higher than today's, 0 otherwise.

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line - signal_line  # MACD histogram

def compute_bollinger_width(series, window=20, num_std=2):
    sma = series.rolling(window).mean()
    std = series.rolling(window).std()
    upper = sma + num_std * std
    lower = sma - num_std * std
    return (upper - lower) / sma  # normalised band width

# Build feature set
data = df.copy()
data["sma_7"] = data["close"].rolling(7).mean()
data["sma_30"] = data["close"].rolling(30).mean()
data["sma_200"] = data["close"].rolling(200).mean()
data["rsi_14"] = compute_rsi(data["close"], 14)
data["macd_hist"] = compute_macd(data["close"])
data["bb_width"] = compute_bollinger_width(data["close"])
data["daily_return"] = data["close"].pct_change()
data["volatility_7d"] = data["daily_return"].rolling(7).std()

# Target: next-day direction (1 = price up, 0 = price down/flat)
data["target"] = (data["close"].shift(-1) > data["close"]).astype(int)

data.dropna(inplace=True)
print(f"Final dataset shape after feature engineering: {data.shape}")
data.head()

In [ ]:
FEATURES = ["sma_7", "sma_30", "sma_200", "rsi_14", "macd_hist",
            "bb_width", "daily_return", "volatility_7d", "volume"]

X = data[FEATURES].values
y = data["target"].values

print(f"X shape: {X.shape} | y shape: {y.shape}")
print(f"Class balance — Up: {y.sum()} ({y.mean():.1%}) | Down: {len(y)-y.sum()} ({1-y.mean():.1%})")

## 4. Train/Test Split — Time Series Aware

**Critical for financial data:** we never shuffle. A random shuffle would let the model "see the future" during training (look-ahead bias), inflating accuracy artificially. We use a chronological split and `TimeSeriesSplit` for cross-validation, exactly as described in the report's methodology.

In [ ]:
# Chronological 80/20 split (no shuffling)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Scale features (important for Logistic Regression and LSTM; trees don't strictly need it but it doesn't hurt)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

# TimeSeriesSplit object for cross-validation, used below
tscv = TimeSeriesSplit(n_splits=5)

## 5. Model 1 — Logistic Regression (Baseline)

A simple, interpretable linear baseline. Every other model must beat this to justify its added complexity.

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression — Test Set Performance")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred_lr):.3f}")
print(f"  Recall:    {recall_score(y_test, y_pred_lr):.3f}")
print(f"  F1-score:  {f1_score(y_test, y_pred_lr):.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba_lr):.3f}")

## 6. Model 2 — Random Forest

An ensemble of decision trees that reduces overfitting via bagging, and gives us feature importance — useful for explaining *why* the model predicts what it predicts (important for the fintech transparency discussion in the report).

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=10,
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train, y_train)  # tree models don't need scaled features

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest — Test Set Performance")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred_rf):.3f}")
print(f"  Recall:    {recall_score(y_test, y_pred_rf):.3f}")
print(f"  F1-score:  {f1_score(y_test, y_pred_rf):.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.3f}")

In [ ]:
# Feature importance — useful for your report's "explainability" discussion
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, color="steelblue")
plt.title("Random Forest — Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("rf_feature_importance.png", dpi=150)
plt.show()

## 7. Model 3 — XGBoost

Gradient boosting, identified in the 2025 literature reviewed in the report as the consistent top performer for Bitcoin direction classification.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=SEED
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost — Test Set Performance")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_xgb):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred_xgb):.3f}")
print(f"  Recall:    {recall_score(y_test, y_pred_xgb):.3f}")
print(f"  F1-score:  {f1_score(y_test, y_pred_xgb):.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.3f}")

## 8. Model 4 — LSTM (Deep Learning)

LSTM captures sequential dependencies that the previous three models cannot — each prediction looks back at the last 10 trading days, not just a single day's snapshot. This is the model most representative of "deep learning methods" requested in the brief.

In [ ]:
# Build sequences of the last N days for each prediction (LSTM needs 3D input: samples, timesteps, features)
def create_sequences(X, y, window=10):
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i:i + window])
        ys.append(y[i + window])
    return np.array(Xs), np.array(ys)

WINDOW = 10
X_seq, y_seq = create_sequences(np.vstack([X_train_scaled, X_test_scaled]),
                                  np.concatenate([y_train, y_test]), WINDOW)

# Re-split chronologically after sequencing
split_seq_idx = len(X_train_scaled) - WINDOW
X_train_seq, X_test_seq = X_seq[:split_seq_idx], X_seq[split_seq_idx:]
y_train_seq, y_test_seq = y_seq[:split_seq_idx], y_seq[split_seq_idx:]

print(f"LSTM input shape — train: {X_train_seq.shape}, test: {X_test_seq.shape}")

In [ ]:
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW, len(FEATURES))),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.15,
    epochs=30,
    batch_size=32,
    verbose=0
)

print("LSTM training complete.")

In [ ]:
y_proba_lstm = lstm_model.predict(X_test_seq, verbose=0).flatten()
y_pred_lstm = (y_proba_lstm > 0.5).astype(int)

print("LSTM — Test Set Performance")
print(f"  Accuracy:  {accuracy_score(y_test_seq, y_pred_lstm):.3f}")
print(f"  Precision: {precision_score(y_test_seq, y_pred_lstm):.3f}")
print(f"  Recall:    {recall_score(y_test_seq, y_pred_lstm):.3f}")
print(f"  F1-score:  {f1_score(y_test_seq, y_pred_lstm):.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test_seq, y_proba_lstm):.3f}")

In [ ]:
# Training history plot — useful evidence of proper training (not overfit) for your appendix
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Loss Curve")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM Accuracy Curve")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.savefig("lstm_training_curves.png", dpi=150)
plt.show()

## 9. Model Comparison

This summary table is the centrepiece evidence for your report's Machine Learning Method and Conclusion sections.

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost", "LSTM"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test_seq, y_pred_lstm),
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test_seq, y_pred_lstm),
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test_seq, y_pred_lstm),
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test_seq, y_pred_lstm),
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb),
        roc_auc_score(y_test_seq, y_proba_lstm),
    ],
}).round(3)

results.to_csv("model_comparison_results.csv", index=False)
results

In [ ]:
# Visual comparison — bar chart for the report/presentation
results_melted = results.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(11, 6))
sns.barplot(data=results_melted, x="Metric", y="Score", hue="Model")
plt.title("Model Performance Comparison — Bitcoin Direction Prediction")
plt.ylim(0, 1)
plt.legend(loc="upper right", bbox_to_anchor=(1.25, 1))
plt.tight_layout()
plt.savefig("model_comparison_chart.png", dpi=150)
plt.show()

In [ ]:
# ROC curves — all four models on one plot, strong visual for your presentation slides
plt.figure(figsize=(7, 7))

for name, y_true, y_proba in [
    ("Logistic Regression", y_test, y_proba_lr),
    ("Random Forest", y_test, y_proba_rf),
    ("XGBoost", y_test, y_proba_xgb),
    ("LSTM", y_test_seq, y_proba_lstm),
]:
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.2f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — All Models")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves_comparison.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrices — one figure, four subplots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

cm_data = [
    ("Logistic Regression", y_test, y_pred_lr),
    ("Random Forest", y_test, y_pred_rf),
    ("XGBoost", y_test, y_pred_xgb),
    ("LSTM", y_test_seq, y_pred_lstm),
]

for ax, (name, y_true, y_pred) in zip(axes, cm_data):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["Down", "Up"], yticklabels=["Down", "Up"])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

## 10. Conclusion of Analysis

Print a plain-language summary you can paste directly into your report's Machine Learning Method / Conclusion sections.

In [ ]:
best_model = results.loc[results["F1-Score"].idxmax(), "Model"]
best_f1 = results["F1-Score"].max()

print("="*60)
print("SUMMARY FOR REPORT")
print("="*60)
print(f"Best performing model (by F1-score): {best_model} ({best_f1:.3f})")
print()
print(results.to_string(index=False))
print()
print("Note: cite these exact figures in your report's Machine Learning")
print("Method and Conclusion sections. Update narrative if your results")
print("differ slightly from the indicative figures already drafted.")

## Appendix: Notes on Reproducibility and Limitations

- **No look-ahead bias:** all splits are chronological; scaler is fit only on training data.
- **Random seed (42)** fixed throughout for reproducibility — re-running this notebook should produce near-identical results (LSTM may vary very slightly due to GPU non-determinism).
- **Limitation:** this is a direction-classification task (up/down), not a magnitude/price-level forecast. Accuracy near 55–62% is consistent with published 2025 literature on this task — Bitcoin direction is inherently close to a difficult, near-random classification problem, which is itself a useful finding to discuss in your report's ethical/limitations reflection.
- **Extending this notebook:** to add sentiment analysis, fetch Twitter/Reddit data and merge on date as an additional feature column before the train/test split.